# From Waveform to Genre — Modeling & Training


**FMA dataset (citation)**  
Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. 18th ISMIR. PDF: https://arxiv.org/pdf/1612.01840.pdf

**License / data note:** FMA metadata is CC BY 4.0; audio files follow per-artist Creative Commons licenses. Consult the original dataset for terms of use.

---

### Project adaptation note
This notebook adapts the FMA data pipeline for the current workflow and interactive environments (Google Colab / Drive). Key adaptations include Colab/Drive path variables, idempotent download & extraction steps, automatic manifest generation for reproducibility, and helper utilities for checksums and seeding. Any reuse of original code or logic from the mdeff/fma project is indicated in the header and documented in the repository.


***
This notebook prepares and trains an audio-based genre classifier end-to-end. It configures the Colab environment and project paths, defines audio preprocessing and dataset utilities, builds a PyTorch dataset/loader that converts raw audio into normalized Mel-spectrogram inputs, and implements the model training loop with checkpointing and training-history logging. After training, it runs inference on the test split, saves per-track predictions and evaluation metrics (confusion matrix, classification report), and produces diagnostics and visualizations — including learning curves, latent-space projections (t-SNE/UMAP) with silhouette scores, class distribution analysis, and example spectrograms for misclassified tracks. Finally, the notebook records provenance metadata and synchronizes the processed data and outputs to Google Drive.

### Short comparison vs. original mdeff/fma
1. Focused on end-to-end model training in Colab (audio waveform → mel-spectrogram → CNN) rather than the original repo's feature-centric tooling.

2. Adds Colab/Drive integration, automated metadata download + SHA-1 verification, balanced per-genre sampling, and stratified train/val/test splits.

3. Implements data provenance saving and Drive sync (overwrites previous copy) for reproducibility.

4. Includes inference, evaluation, visualization (t-SNE/UMAP, confusion matrices, learning curves) and error-analysis pipelines not present in the original.

### Rationale for Data Transfer Methodology

This notebook utilizes a combination of native shell commands (Bash) and Python scripts to manage data transfer and preparation. This hybrid approach is selected based on the specific requirements of handling large audio datasets within the Colab environment.

- **System Performance and Resource Efficiency:** Bash-based utilities such as `cp`, `rsync`, and `unzip` are employed for high-volume binary I/O tasks. These tools operate at the system level, providing high execution speeds and minimal memory overhead by streaming data outside of the Python runtime.
- **Integrity Verification:** The use of shell commands like `unzip -t` allows for efficient verification of archive integrity. Combined with standard shell error handling (`set -euo pipefail`), this ensures that issues are identified early through clear exit codes.
- **Programmatic Flexibility:** Python is utilized for tasks requiring complex logic, such as HTTP requests with retry mechanisms, dynamic file indexing, and integration with PyTorch DataLoaders. It provides a structured environment for logging and exception handling.
- **Environment Integration:** While shell operations are effective for bulk data movement (e.g., extracting multiple gigabytes), Python is integrated directly with the project’s classes and analysis tools.

**Implementation Standard:**
- **Bash:** Recommended for non-iterative, heavy I/O operations including archive retrieval, integrity testing, extraction, and directory organization.
- **Python:** Recommended for logic-dependent operations such as track ID filtering, data transformations, and the generation of training artifacts with associated metadata.
- **Documentation:** Each code cell includes a brief indication of the chosen method to maintain technical clarity.

This methodology ensures that file handling is both performant and reliable, maintaining a manageable and reproducible data pipeline.

### Mount Drive and Configure Project Paths
This mounts Google Drive in Colab, sets up project and data directories, ensures the local data folder exists, changes the working directory to the project folder, and prints the local data path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
WORKDIR = "/content/waveform_genre_project"
LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
DRIVE_ROOT = "/content/drive/MyDrive/waveform_analysis_outputs"
os.makedirs(LOCAL_DATA, exist_ok=True)
os.chdir(WORKDIR)
print("WORKDIR set, LOCAL_DATA:", LOCAL_DATA)

Sets up required libraries and a compact modeling configuration (audio paths and preprocessing params, random seed, training hyperparameters, and output file locations) for training an audio genre classifier.

In [ ]:
# standard libs for filesystem, archives, downloads, and utility operations
import os, zipfile, urllib.request, shutil, time, datetime
# data libraries for arrays and tabular data
import numpy as np, pandas as pd
# PyTorch for model definition and training
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# audio processing library used to load and convert audio to spectrograms
import librosa

# Modeling config (kept compact)
# URL to the small FMA audio archive (download source for audio files)
AUDIO_URL = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
# local path where the downloaded audio ZIP will be stored
AUDIO_ZIP_LOCAL = os.path.join(LOCAL_DATA, "fma_small.zip")
AUDIO_DIR = os.path.join(LOCAL_DATA, "audio")
# audio preprocessing parameters: sampling rate and clip length (seconds)
SAMPLE_RATE = 22050
CLIP_SECONDS = 10
# spectrogram parameters: number of mel bands, FFT window size, and hop length
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
# reproducibility: fixed random seed for sampling/splitting/initialization
SEED = 42
# training hyperparameters: batch size, learning rate, and number of epochs
BATCH_SIZE = 16
LR = 5e-4
EPOCHS = 1   # demo short training by default
# paths for saving model weights and training history (CSV)
MODEL_PATH = os.path.join(LOCAL_DATA, "genre_classifier_net.pt")
HISTORY_CSV = os.path.join(LOCAL_DATA, "training_history.csv")

### Automated Dataset Restoration from Google Drive

Description: This code automates the restoration of the workspace database by intelligently searching for the most recent copy of the data_storage folder or the train.csv file within Google Drive and copying it to the local Colab environment to avoid re-generating the data.

In [ ]:
# single-cell solution: find data_storage ONLY within waveform_analysis_outputs in Drive and copy to local WORKDIR
from pathlib import Path
import os, shutil, glob, sys

WORKDIR = "/content/waveform_genre_project"
LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
DRIVE_ROOT = "/content/drive/MyDrive"
SEARCH_ROOT = os.path.join(DRIVE_ROOT, "waveform_analysis_outputs")  # search is limited to this directory

# 1) Mount Drive if needed (Colab)
if not os.path.exists("/content/drive"):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped or failed (not Colab?). Continue if Drive already mounted.")
else:
    # quick check whether MyDrive exists
    if not os.path.exists(DRIVE_ROOT):
        print("Warning: /content/drive/MyDrive not found. If you mounted Drive under a different path, adjust DRIVE_ROOT.")

# 2) Ensure WORKDIR exists and switch to it
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("Working dir:", os.getcwd())

# 3) If local data already present and has train.csv -> done
local_train = os.path.join(LOCAL_DATA, "train.csv")
if os.path.exists(local_train):
    print("Local train.csv already present at", local_train)
    print("You can continue running the notebook.")
    pass

# 4) Search for data_storage folders ONLY under waveform_analysis_outputs (recursive)
candidates = glob.glob(os.path.join(SEARCH_ROOT, "**", "data_storage"), recursive=True)
candidates = [c for c in candidates if os.path.isdir(c)]
print("Found data_storage candidate folders in Drive (within waveform_analysis_outputs):", len(candidates))

def score_path(p):
    # score by number of files (bigger is better) and mtime
    try:
        n_files = sum(1 for _ in Path(p).rglob('*') if _.is_file())
        mtime = Path(p).stat().st_mtime
        return (n_files, mtime)
    except Exception:
        return (0, 0)

# 5) If candidates found -> pick best and copy
if candidates:
    scored = [(score_path(p), p) for p in candidates]
    scored.sort(reverse=True)  # prefer more files / newer
    best = scored[0][1]
    print("Selected candidate to copy:", best)
    # copytree with dirs_exist_ok (Py3.8+). If older, do safe fallback.
    try:
        shutil.copytree(best, LOCAL_DATA, dirs_exist_ok=True)
    except TypeError:
        # older shutil (unlikely) -> manual copy
        for root, dirs, files in os.walk(best):
            rel = os.path.relpath(root, best)
            target_dir = os.path.join(LOCAL_DATA, rel) if rel != "." else LOCAL_DATA
            os.makedirs(target_dir, exist_ok=True)
            for f in files:
                srcf = os.path.join(root, f)
                dstf = os.path.join(target_dir, f)
                if not os.path.exists(dstf):
                    shutil.copy2(srcf, dstf)
    print("Copy complete. Local data_storage now at:", LOCAL_DATA)
    if os.path.exists(local_train):
        print("train.csv found locally -> OK. You can re-run the cell that failed earlier.")
    else:
        print("Warning: train.csv not found inside the copied data_storage. List files in LOCAL_DATA root:")
        print(os.listdir(LOCAL_DATA)[:50])
    pass

# 6) If no data_storage folder: search for any train.csv ONLY under waveform_analysis_outputs
found_trains = glob.glob(os.path.join(SEARCH_ROOT, "**", "train.csv"), recursive=True)
if found_trains:
    # choose the first or best candidate (we pick the one with largest size)
    found_trains = sorted(found_trains, key=lambda p: os.path.getsize(p) if os.path.exists(p) else 0, reverse=True)
    train_path = found_trains[0]
    parent = os.path.dirname(train_path)
    print("Found train.csv at:", train_path)
    print("Copying its parent folder:", parent, "->", LOCAL_DATA)
    try:
        shutil.copytree(parent, LOCAL_DATA, dirs_exist_ok=True)
    except TypeError:
        for root, dirs, files in os.walk(parent):
            rel = os.path.relpath(root, parent)
            target_dir = os.path.join(LOCAL_DATA, rel) if rel != "." else LOCAL_DATA
            os.makedirs(target_dir, exist_ok=True)
            for f in files:
                srcf = os.path.join(root, f)
                dstf = os.path.join(target_dir, f)
                if not os.path.exists(dstf):
                    shutil.copy2(srcf, dstf)
    print("Copy complete. Check:", os.path.join(LOCAL_DATA, "train.csv"))
    if os.path.exists(local_train):
        print("train.csv local copy OK. You can re-run the failing cell.")
    else:
        print("train.csv still not found after copy; list LOCAL_DATA contents:", os.listdir(LOCAL_DATA)[:50])
    pass
else:
    # 7) Nothing found in Drive under waveform_analysis_outputs
    print("No data_storage folder and no train.csv found under", SEARCH_ROOT)

Verifies that the local data directory is available by listing its contents, loading train.csv into a pandas DataFrame, and printing the total row count along with a preview of the first few rows.

In [ ]:
# check that local dialing is available
import os, pandas as pd
LOCAL_DATA = "/content/waveform_genre_project/data_storage"
print("Local data dir:", LOCAL_DATA)
print("Top-level files:", os.listdir(LOCAL_DATA)[:50])
df = pd.read_csv(os.path.join(LOCAL_DATA, "train.csv"))
print("train.csv rows:", len(df))
print(df.head())

Еxtracts the required MP3 files from a large ZIP archive, resolves the file paths, and loads 10-second audio clips, converting them into normalized Mel spectrograms ready for the model.

In [ ]:
# build the filesystem path to an audio file given a numeric track id
def track_path(track_id):
    # format track_id as a zero-padded 6-digit string (e.g., 42 -> "000042")
    s = f"{int(track_id):06d}"
    # return path AUDIO_DIR/<first3>/<000042.mp3>
    return os.path.join(AUDIO_DIR, s[:3], s + ".mp3")

# ensure requested audio files are available locally by extracting them from the large ZIP
def prepare_audio_for(ids):
    os.makedirs(AUDIO_DIR, exist_ok=True)
    # download big zip if not present (warning: large)
    if not os.path.exists(AUDIO_ZIP_LOCAL):
        print("Downloading fma_small.zip (~7.2GB). This may take long.")
        urllib.request.urlretrieve(AUDIO_URL, AUDIO_ZIP_LOCAL)
        print("Download done.")
    # open the ZIP archive for reading
    with zipfile.ZipFile(AUDIO_ZIP_LOCAL, "r") as zf:
        # cache the list of members for quick membership tests
        members = set(zf.namelist())
        # Iterate over requested track ids
        for tid in ids:
            s = f"{int(tid):06d}"
            member = f"fma_small/{s[:3]}/{s}.mp3"
            # If the file is not present in the archive, skip it
            if member not in members:
                # skip missing
                continue
            outdir = os.path.join(AUDIO_DIR, s[:3]); os.makedirs(outdir, exist_ok=True)
            dst = os.path.join(outdir, s + ".mp3")
            if not os.path.exists(dst):
                with zf.open(member) as src, open(dst, "wb") as out:
                    out.write(src.read())

# Load audio for a single track and convert to a normalized Mel spectrogram (float32)
def load_mel_from_track(track_id):
    path = track_path(track_id)
    # load the audio clip, resampling to SAMPLE_RATE and truncating/prefetching up to CLIP_SECONDS
    y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=CLIP_SECONDS)
    target_len = SAMPLE_RATE * CLIP_SECONDS
    # pad with zeros or trim so the signal length equals target_len
    y = np.pad(y, (0, max(0, target_len - len(y))))[:target_len]
    # compute the Mel spectrogram (power) with configured parameters
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    # convert power spectrogram to decibel (log) scale referenced to the max value
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    # return as float32
    return mel_db.astype(np.float32)

### PyTorch Dataset and SmallCNN Implementation
Defines a PyTorch Dataset that loads normalized Mel-spectrograms and genre labels from a CSV, and a compact CNN that maps those spectrograms to class scores (with an option to return the intermediate embedding).

In [ ]:
# dataset that reads a CSV of track_ids and genres and returns (input_tensor, label)
class AudioDataset(Dataset):
    def __init__(self, csv_path, genre_to_idx):
        # load CSV into a DataFrame and store mapping from genre name -> index
        self.df = pd.read_csv(csv_path)
        self.genre_to_idx = genre_to_idx
    def __len__(self):
      # number of examples
      return len(self.df)
    def __getitem__(self, i):
        # read the i-th row (contains track_id and genre)
        row = self.df.iloc[i]
        # load and preprocess audio into a Mel spectrogram (numpy array): shape (n_mels, time)
        mel = load_mel_from_track(int(row["track_id"]))
        # convert to a torch tensor and add channel dim -> (1, n_mels, time)
        x = torch.from_numpy(mel).unsqueeze(0)
        # map genre string to integer class label
        y = self.genre_to_idx[row["genre"]]
        return x, int(y)

# small convolutional neural network for Mel-spectrogram input
class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        # feature extractor: three conv blocks, downsamples via MaxPool, ends with global pooling
        self.features = nn.Sequential(
            # conv block 1: input 1 channel -> 16 channels, keeps spatial size with padding
            nn.Conv2d(1,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2), # halves time and freq dims
            # conv block 2: 16 -> 32 channels
            nn.Conv2d(16,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2), # further downsampling
            # conv block 3: 32 -> 64 channels, then adaptive pool to 1x1 spatially
            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # outputs shape (batch, 64, 1, 1)
            nn.AdaptiveAvgPool2d(1)
        )
        # dropout for regularization before the classification head
        self.drop = nn.Dropout(0.3)
        # final linear layer mapping 64-d embedding -> number of classes
        self.head = nn.Linear(64, n_classes)
    def forward(self,x, return_embedding=False):
        # extract features and flatten to (batch, 64)
        f = self.features(x).flatten(1)
        f = self.drop(f)
        # If requested, return the embedding vector (useful for t-SNE/UMAP or metric learning)
        if return_embedding:
            return f
        # otherwise return raw class logits
        return self.head(f)

Reads the train/validation/test CSVs, builds a genre→index map, optionally prepares audio, constructs PyTorch Dataset/DataLoader objects, and selects the compute device.

In [ ]:
# paths to the CSV files for each split
train_csv = os.path.join(LOCAL_DATA, "train.csv")
val_csv = os.path.join(LOCAL_DATA, "val.csv")
test_csv = os.path.join(LOCAL_DATA, "test.csv")

# load the training CSV to discover the set of genres
train_df = pd.read_csv(train_csv)

# create a sorted list of unique genres and a mapping from genre name to integer index
genres = sorted(train_df["genre"].unique())
g2i = {g:i for i,g in enumerate(genres)}
# collect ids and prepare audio
all_ids = pd.concat([
    pd.read_csv(train_csv)["track_id"],
    pd.read_csv(val_csv)["track_id"],
    pd.read_csv(test_csv)["track_id"]
    ]).tolist()
# Prepare audio (comment out if audio already present)
# prepare_audio_for(all_ids)

# create Dataset instances for training and validation using the genre->index mapping
train_ds = AudioDataset(train_csv, g2i)
val_ds = AudioDataset(val_csv, g2i)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)

# select CUDA if available, otherwise fall back to CPU, and print chosen device and class map
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "classes:", g2i)

The script searches for existing processed data in Google Drive; if none is found, it attempts to locate or download the missing audio archive (fma_small.zip) before copying and synchronizing the necessary files into the local workspace for Colab operation.

In [ ]:
# if necessary
#
# # Single-cell solution: find data_storage ONLY within waveform_analysis_outputs in Drive and copy to local WORKDIR
# # plus: if zip is missing — downloads it from the internet, copies zip to Drive
# from pathlib import Path
# import os, shutil, glob, sys
# import zipfile
# from tqdm.auto import tqdm

# # --- Settings ---
# WORKDIR = "/content/waveform_genre_project"
# LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
# DRIVE_ROOT = "/content/drive/MyDrive"
# SEARCH_ROOT = os.path.join(DRIVE_ROOT, "waveform_analysis_outputs")  # Search is limited to this directory

# # ZIP download / extract settings
# AUDIO_URL = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"  # URL to audio archive
# LOCAL_ZIP_NAME = "fma_small.zip"
# PROJECT_DIR = WORKDIR  # where to extract the archive
# SRC_CANDIDATES = [os.path.join("/content", LOCAL_ZIP_NAME), os.path.join(WORKDIR, "data_storage", LOCAL_ZIP_NAME)]
# DRIVE_TARGET_DIR = SEARCH_ROOT
# CHUNK = 4 * 1024 * 1024  # 4 MB

# # --- 1) Mount Drive if needed (Colab) ---
# if not os.path.exists("/content/drive"):
#     try:
#         from google.colab import drive
#         drive.mount('/content/drive', force_remount=False)
#     except Exception as e:
#         print("Drive mount skipped or failed (not Colab?). Continue if Drive already mounted.")
# else:
#     if not os.path.exists(DRIVE_ROOT):
#         print("Warning: /content/drive/MyDrive not found. If you mounted Drive under a different path, adjust DRIVE_ROOT.")

# # --- 2) ensure WORKDIR exists and switch to it ---
# os.makedirs(WORKDIR, exist_ok=True)
# os.chdir(WORKDIR)
# print("Working dir:", os.getcwd())

# # --- 3) If local data already present and has train.csv -> done ---
# local_train = os.path.join(LOCAL_DATA, "train.csv")
# if os.path.exists(local_train):
#     print("Local train.csv already present at", local_train)
#     print("You can continue running the notebook.")

# # --- 4) search for data_storage folders ONLY under waveform_analysis_outputs (recursive) ---
# candidates = glob.glob(os.path.join(SEARCH_ROOT, "**", "data_storage"), recursive=True)
# candidates = [c for c in candidates if os.path.isdir(c)]
# print("Found data_storage candidate folders in Drive (within waveform_analysis_outputs):", len(candidates))

# def score_path(p):
#     try:
#         n_files = sum(1 for _ in Path(p).rglob('*') if _.is_file())
#         mtime = Path(p).stat().st_mtime
#         return (n_files, mtime)
#     except Exception:
#         return (0, 0)

# # 5) If candidates found -> pick best and copy
# if candidates:
#     scored = [(score_path(p), p) for p in candidates]
#     scored.sort(reverse=True)
#     best = scored[0][1]
#     print("Selected candidate to copy:", best)
#     try:
#         shutil.copytree(best, LOCAL_DATA, dirs_exist_ok=True)
#     except TypeError:
#         for root, dirs, files in os.walk(best):
#             rel = os.path.relpath(root, best)
#             target_dir = os.path.join(LOCAL_DATA, rel) if rel != "." else LOCAL_DATA
#             os.makedirs(target_dir, exist_ok=True)
#             for f in files:
#                 srcf = os.path.join(root, f)
#                 dstf = os.path.join(target_dir, f)
#                 if not os.path.exists(dstf):
#                     shutil.copy2(srcf, dstf)
#     print("Copy complete. Local data_storage now at:", LOCAL_DATA)
#     if os.path.exists(local_train):
#         print("train.csv found locally -> OK. You can re-run the cell that failed earlier.")
#     else:
#         print("Warning: train.csv not found inside the copied data_storage. List files in LOCAL_DATA root:")
#         print(os.listdir(LOCAL_DATA)[:50])

# # --- 6) If no data_storage folder: search for any train.csv ONLY under waveform_analysis_outputs ---
# found_trains = glob.glob(os.path.join(SEARCH_ROOT, "**", "train.csv"), recursive=True)
# if found_trains:
#     found_trains = sorted(found_trains, key=lambda p: os.path.getsize(p) if os.path.exists(p) else 0, reverse=True)
#     train_path = found_trains[0]
#     parent = os.path.dirname(train_path)
#     print("Found train.csv at:", train_path)
#     print("Copying its parent folder:", parent, "->", LOCAL_DATA)
#     try:
#         shutil.copytree(parent, LOCAL_DATA, dirs_exist_ok=True)
#     except TypeError:
#         for root, dirs, files in os.walk(parent):
#             rel = os.path.relpath(root, parent)
#             target_dir = os.path.join(LOCAL_DATA, rel) if rel != "." else LOCAL_DATA
#             os.makedirs(target_dir, exist_ok=True)
#             for f in files:
#                 srcf = os.path.join(root, f)
#                 dstf = os.path.join(target_dir, f)
#                 if not os.path.exists(dstf):
#                     shutil.copy2(srcf, dstf)
#     print("Copy complete. Check:", os.path.join(LOCAL_DATA, "train.csv"))
#     if os.path.exists(local_train):
#         print("train.csv local copy OK. You can re-run the failing cell.")
#     else:
#         print("train.csv still not found after copy; list LOCAL_DATA contents:", os.listdir(LOCAL_DATA)[:50])
# else:
#     print("No data_storage folder and no train.csv found under", SEARCH_ROOT)

# # --- 7) ZIP handling: find local zip or download from internet ---
# # Find existing zip
# src_zip = None
# for p in SRC_CANDIDATES:
#     if os.path.exists(p):
#         src_zip = p
#         break

# # If not found, search /content
# if src_zip is None:
#     for root, dirs, files in os.walk("/content"):
#         if LOCAL_ZIP_NAME in files:
#             src_zip = os.path.join(root, LOCAL_ZIP_NAME)
#             break

# # If still not found, try to download if AUDIO_URL is set
# if src_zip is None:
#     if AUDIO_URL:
#         try:
#             import requests
#             print("Започвам сваляне от:", AUDIO_URL)
#             # stream download with progress
#             resp = requests.get(AUDIO_URL, stream=True, timeout=60)
#             resp.raise_for_status()
#             total = int(resp.headers.get('content-length', 0))
#             dst = os.path.join("/content", LOCAL_ZIP_NAME)
#             with open(dst, "wb") as f, tqdm(total=total, unit='B', unit_scale=True, desc="Сваляне zip") as pbar:
#                 for chunk in resp.iter_content(chunk_size=CHUNK):
#                     if chunk:
#                         f.write(chunk)
#                         pbar.update(len(chunk))
#             src_zip = dst
#             print("Свалянето приключи:", src_zip)
#         except Exception as e:
#             print("Свалянето от интернет не успя:", e)
#             src_zip = None
#     else:
#         print("Няма локален zip и няма зададен валиден AUDIO_URL. Пропускам сваляне.")

# if src_zip is None:
#     print("Не открих zip за разархивиране. Прекратено.")
# else:
#     print("Намерих ZIP:", src_zip)

#     # --- 8) Copy zip to Drive target dir (if Drive mounted) ---
#     if os.path.exists(DRIVE_ROOT):
#         os.makedirs(DRIVE_TARGET_DIR, exist_ok=True)
#         dst_drive = os.path.join(DRIVE_TARGET_DIR, os.path.basename(src_zip))
#         try:
#             if os.path.exists(dst_drive) and os.path.getsize(dst_drive) == os.path.getsize(src_zip):
#                 print("Архивът вече е копиран в Drive:", dst_drive)
#             else:
#                 print(f"Копиране на {src_zip} -> {dst_drive} ...")
#                 total = os.path.getsize(src_zip)
#                 with open(src_zip, "rb") as fr, open(dst_drive, "wb") as fw, tqdm(total=total, unit='B', unit_scale=True, desc="Копиране в Drive") as pbar:
#                     while True:
#                         chunk = fr.read(CHUNK)
#                         if not chunk:
#                             break
#                         fw.write(chunk)
#                         pbar.update(len(chunk))
#                 print("Копиран в Drive:", dst_drive)
#         except Exception as e:
#             print("Копиране в Drive се провали:", e)
#     else:
#         print("Google Drive не е наличен/монтиран — пропускам копиране в Drive.")

The script locates fma_small.zip (locally or in Drive), extracts it (defaulting to .mp3 files only) into the target folder with a progress bar, skips already existing files, and reports the number of successfully extracted, skipped, and failed files.

In [ ]:
# unzip fma_small.zip -> /content/waveform_genre_project/data_storage/audio (only .mp3 files, with progress)
import os
import zipfile
from pathlib import Path
from tqdm.auto import tqdm

# settings
ZIP_CANDIDATES = [
    "/content/fma_small.zip",
    "/content/waveform_genre_project/data_storage/fma_small.zip",
    "/content/drive/MyDrive/waveform_analysis_outputs/fma_small.zip",
    "/content/drive/MyDrive/fma_small.zip"
]
TARGET_DIR = "/content/waveform_genre_project/data_storage/audio"
CHUNK = 4 * 1024 * 1024  # 4 MB
EXTRACT_ONLY_MP3 = True  # True = only .mp3 files; False = all files

# find available zip
zip_path = None
for p in ZIP_CANDIDATES:
    if os.path.exists(p):
        zip_path = p
        break

# if not found - search recursively under /content
if zip_path is None:
    for root, dirs, files in os.walk("/content"):
        if "fma_small.zip" in files:
            zip_path = os.path.join(root, "fma_small.zip")
            break

if zip_path is None:
    raise FileNotFoundError("I couldn't find fma_small.zip in the expected locations (/content, data_storage or Drive). Place the archive in /content or data_storage and run it again.")

print("I will unzip:", zip_path)
os.makedirs(TARGET_DIR, exist_ok=True)

# Отваряне и извличане (само .mp3 по подразбиране)
with zipfile.ZipFile(zip_path, 'r') as z:
    bad = z.testzip()
    if bad is not None:
        raise zipfile.BadZipFile(f"Archive is corrupted - first problematic member: {bad}")

    members = z.namelist()
    # Филтрирай късните (файлови) записи според желанието
    file_members = [m for m in members if not m.endswith('/')]
    if EXTRACT_ONLY_MP3:
        file_members = [m for m in file_members if m.lower().endswith('.mp3')]

    if not file_members:
        print("No files found to extract (according to settings).")
    else:
        # премахване на общ топ-level префикс (ако има) за по-чисти пътища
        def common_prefix(paths):
            if not paths:
                return ""
            split_paths = [p.split('/') for p in paths]
            prefix = []
            for parts in zip(*split_paths):
                if all(p == parts[0] for p in parts):
                    prefix.append(parts[0])
                else:
                    break
            return "/".join(prefix) + ("/" if prefix else "")

        prefix = common_prefix([p for p in file_members][:500])
        if prefix:
            print("Common prefix found in archive:", repr(prefix))

        # общ брой байтове за прогрес
        infos = [z.getinfo(m) for m in file_members]
        total_bytes = sum(info.file_size for info in infos)
        print(f"Number of files to extract: {len(file_members)}, total ~{total_bytes // (1024**2)} MB")

        extracted = 0
        skipped = 0
        errors = 0

        with tqdm(total=total_bytes, unit='B', unit_scale=True, desc="Unzip (bytes)") as pbar:
            for m in file_members:
                rel = m[len(prefix):] if prefix and m.startswith(prefix) else m
                dst_path = os.path.join(TARGET_DIR, rel)
                # създай целевата папка
                os.makedirs(os.path.dirname(dst_path), exist_ok=True)

                try:
                    info = z.getinfo(m)
                except KeyError:
                    # някакъв странен entry
                    errors += 1
                    continue

                # прескачане ако файлът вече съществува със същия размер
                if os.path.exists(dst_path) and os.path.getsize(dst_path) == info.file_size:
                    skipped += 1
                    pbar.update(info.file_size)
                    continue

                # записвай на части и обновявай pbar
                try:
                    with z.open(m) as srcf, open(dst_path, "wb") as dstf:
                        while True:
                            chunk = srcf.read(CHUNK)
                            if not chunk:
                                break
                            dstf.write(chunk)
                            pbar.update(len(chunk))
                    extracted += 1
                except Exception as e:
                    print(f"Retrieval error {m}: {e}")
                    errors += 1

        print(f"Done. Retrieved: {extracted}, skipped: {skipped}, errors: {errors}")
        # кратка проверка: покажи първите няколко mp3 пътища
        found = []
        for root, dirs, files in os.walk(TARGET_DIR):
            for f in files:
                if f.lower().endswith(".mp3"):
                    found.append(os.path.join(root, f))
                    if len(found) >= 10:
                        break
            if len(found) >= 10:
                break
        if found:
            print("First mp3 found (in TARGET_DIR):")
            for p in found[:10]:
                print(" ", p)
        else:
            print("I didn't find the mp3 after unzipping — check the contents of the archive.")

### Optimized Batch Processing with Bash
Using Bash makes large file operations (copying, testing, and extraction) simpler, faster, and ensures reliable error handling.


**Advantages:** Speed, Stability for large files, Reliability, and Direct integrity verification.


**Script Summary:** The script retrieves the fma_small.zip archive from the specified Google Drive location, copies it locally, verifies the archive's integrity, extracts it into the target folder, and prints the total count of extracted MP3 files.

In [ ]:
# copy of the archive
%%bash
set -euo pipefail

ZIP="/content/drive/MyDrive/waveform_analysis_outputs/fma_small.zip"
LOCAL="/content/fma_small.zip"
DEST="/content/waveform_genre_project/data_storage/audio"

echo "Drive ZIP: $ZIP"
if [ ! -f "$ZIP" ]; then
  echo "ERROR: ZIP not found in Drive: $ZIP" >&2
  exit 2
fi

echo "Copy locally..."
cp -v "$ZIP" "$LOCAL"

echo "Testing the archive..."
if unzip -t "$LOCAL" >/dev/null 2>&1; then
  echo "ZIP test: OK"
else
  echo "ZIP test: FAILED (archive is corrupted). I will redownload." >&2
  exit 3
fi

echo "Unzip to $DEST ..."
mkdir -p "$DEST"
unzip -q "$LOCAL" -d "$DEST"

echo "Number of mp3 files:"
find "$DEST" -type f -iname "*.mp3" | wc -l
echo "Done. Files extracted to $DEST"

The script verifies the existence of the specified ZIP archive, creates the target folder, extracts all members with a visual progress bar, and finally counts and displays the found MP3 files.

In [ ]:
# unzip the archive
import os
import zipfile
import shutil
from tqdm.auto import tqdm

# Road settings
zip_path = "/content/fma_small.zip"
extract_path = "/content/waveform_genre_project/data_storage/audio"

# 1. Checking if the zip exists
if not os.path.exists(zip_path):
    print(f"Archive {zip_path} not found! Check the previous step.")
else:
    print(f"Starting to unzip {zip_path}...")
    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        files = zf.namelist()
        # use tqdm for visual progress
        for file in tqdm(files, desc="Unzip"):
            try:
                zf.extract(file, extract_path)
            except Exception as e:
                print(f"Error at {file}: {e}")

    print(f"\nDone! Files extracted to: {extract_path}")

    # content verification
    mp3_count = sum([len(files) for r, d, files in os.walk(extract_path) if any(f.endswith('.mp3') for f in files)])
    print(f"Found mp3 files: {mp3_count}")

The snippet traverses the specified directory and prints the full paths of the first five MP3 files found, or reports that none were found.

In [ ]:
import fnmatch

# directory to search for mp3 files
TARGET_DIR = "/content/waveform_genre_project/data_storage/audio"
# collect up to 5 mp3 file paths
found = []
for root, dirs, files in os.walk(TARGET_DIR):
    # filter the file list to only those matching the pattern "*.mp3"
    for name in fnmatch.filter(files, "*.mp3"):
        found.append(os.path.join(root, name))
        # stop when we have enough examples to avoid scanning the whole tree
        if len(found) >= 5:
            break
    if len(found) >= 5:
        break

if found:
    print("Found mp3s (first 5):")
    for p in found:
        print(p)
else:
    print("No mp3s found in", TARGET_DIR)

The code ensures that the training DataLoaders use the new audio folder (AUDIO_ROOT) by attempting to update already loaded datasets in-place or providing a template for creating new DataLoaders, followed by a quick check for the presence of at least one MP3 file.

In [ ]:
import os, glob, itertools

WORKDIR = "/content/waveform_genre_project"
AUDIO_ROOT = os.path.join(WORKDIR, "data_storage", "audio")
print("AUDIO_ROOT set to:", AUDIO_ROOT)

# 1) If train_dl/val_dl have already been created - try updating their dataset in place
try:
    for dl_name in ("train_dl", "val_dl"):
        dl = globals().get(dl_name, None)
        if dl is None:
            continue
        ds = getattr(dl, "dataset", None)
        if ds is None:
            continue

        # try a few common attributes
        if hasattr(ds, "audio_root"):
            ds.audio_root = AUDIO_ROOT
        if hasattr(ds, "root"):
            ds.root = os.path.join(WORKDIR, "data")
        # if Dataset has a method to recreate an index/list of files, call it
        if hasattr(ds, "build_index"):
            try:
                ds.build_index()
            except Exception as e:
                print(f"build_index failed for {dl_name}: {e}")
        # try to recreate common fields like samples/files/paths
        files_list = []
        for root, _, files in os.walk(AUDIO_ROOT):
            for f in files:
                if f.lower().endswith(".mp3"):
                    files_list.append(os.path.join(root, f))
        for attr in ("samples", "files", "file_list", "filepaths", "paths"):
            if hasattr(ds, attr):
                try:
                    setattr(ds, attr, files_list)
                except Exception as e:
                    print(f"Could not set {attr} on dataset for {dl_name}: {e}")

    print("Updated existing datasets (if present).")
except Exception as e:
    print("Error while updating existing dataloaders:", e)

# 2) If you don't have train_dl/val_dl or want to recreate them:
# Replace MyDataset(...) with your Dataset class/arguments.
if "train_dl" not in globals() or "val_dl" not in globals():
    print("train_dl/val_dl not found — създай ги с AUDIO_ROOT.")
    try:
        from torch.utils.data import DataLoader
        # Example: train_ds = MyDataset(audio_root=AUDIO_ROOT, split="train", transform=...)
        # val_ds   = MyDataset(audio_root=AUDIO_ROOT, split="val", transform=...)
        # train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
        # val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        print("Adapt the example above to your own Dataset and run again.")
    except Exception as e:
        print("Failed when trying to create a DataLoader template:", e)

# 3) quick check that at least one mp3 is available on the new path
example = None
for p in glob.glob(os.path.join(AUDIO_ROOT, "**", "*.mp3"), recursive=True):
    example = p
    break
if example:
    print("Sample mp3 found:", example)
else:
    print("No mp3 found under", AUDIO_ROOT)

This Bash script reorganizes the directory structure by moving the contents of the nested fma_small folder directly into the main audio directory, removes empty folders, and confirms the operation by checking for a specific test file.

In [ ]:
%%bash
# paths
BASE="/content/waveform_genre_project/data_storage/audio"
SRC="$BASE/fma_small"
TEST_FILE="091/091164.mp3"

# 1) show first few subdirectories (check)
ls -1 "$SRC" | head -n 10

# 2) move all files and folders from fma_small to the parent audio/
# If there are a large amount of files, we use rsync (more reliable)
rsync -av --remove-source-files "$SRC"/ "$BASE"/

# 3) delete empty directories in src (if any remain)
find "$SRC" -type d -empty -delete

# 4) remove the empty fma_small folder itself
rmdir "$SRC" 2>/dev/null || true

# 5) check: whether the sample file is already in the expected location
if [ -f "$BASE/$TEST_FILE" ]; then
  echo "OK: файлът е на $BASE/$TEST_FILE"
else
  echo "NOT FOUND: $BASE/$TEST_FILE"
  echo "Списък на първите файлове в $BASE:"
  find "$BASE" -maxdepth 2 -type f | head -n 20
fi

### Model Training and Evaluation Loop
This script defines the complete training and validation loop for the model. It includes initializing the optimizer and the loss function, iterating through epochs, measuring accuracy and loss in real-time, and finally saving the trained model weights and the training history into a CSV file.

In [ ]:
# Initialize the model, optimizer (Adam), and loss function (CrossEntropy)
model = SmallCNN(len(genres)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()
history = []
# main training loop
for epoch in range(1, EPOCHS+1):
    # training Phase
    model.train()
    total_loss = 0; correct = 0; total = 0
    t0 = time.time()
    for x,y in train_dl:
        # move data to GPU/CPU
        x,y = x.to(device), y.to(device)
        # reset gradients, forward pass, calculate loss, and update weights
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward(); opt.step()
        # accumulate metrics
        total += x.size(0)
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
    train_loss = total_loss/total; train_acc = correct/total
    # validation Phase
    model.eval()
    vloss = 0; vcorrect = 0; vtotal = 0
    # disable gradient calculation for efficiency
    with torch.no_grad():
        for x,y in val_dl:
            x,y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out,y)
            vtotal += x.size(0)
            vloss += loss.item() * x.size(0)
            vcorrect += (out.argmax(1) == y).sum().item()
    val_loss = vloss/vtotal; val_acc = vcorrect/vtotal
    epoch_time = time.time()-t0
    # store and print results for the current epoch
    history.append({"epoch":epoch,"train_loss":train_loss,"train_acc":train_acc,"val_loss":val_loss,"val_acc":val_acc,"time_s":epoch_time})
    print(f"Epoch {epoch}: train_acc={train_acc:.3f} val_acc={val_acc:.3f} time={epoch_time:.1f}s")
# save model and history
torch.save(model.state_dict(), MODEL_PATH)
# export training history to a CSV file for future analysis
import pandas as pd
pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)
print("Saved model to", MODEL_PATH)

The code executes a complete PyTorch training loop—it initializes the model, optimizer, and loss function, performs training and validation across several epochs while accumulating metrics and tracking epoch duration, and finally saves the model weights and training history to a CSV file.

In [ ]:
import time
import torch
import torch.nn as nn
import pandas as pd

EPOCHS = 10

# Instantiate model, optimizer and loss function
# Ensure SmallCNN supports your input shape (e.g., 1 channel for spectrograms)
model = SmallCNN(len(genres)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

history = []

print(f"Starting training for {EPOCHS} epochs on {device}...")

for epoch in range(1, EPOCHS + 1):
    # set model to training mode
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    # mark epoch start time
    t0 = time.time()

    for batch_idx, (x, y) in enumerate(train_dl):
        # x shape: [batch, channels, freq, time]
        x, y = x.to(device), y.to(device)

        opt.zero_grad()

        try:
            out = model(x)          # forward pass
            loss = loss_fn(out, y)  # compute loss
            loss.backward()         # backpropagate
            opt.step()              # optimizer step / update weights

            # # Accumulate statistics for the epoch
            batch_size = x.size(0)
            total += batch_size
            total_loss += loss.item() * batch_size
            correct += (out.argmax(1) == y).sum().item()

        except RuntimeError as e:
            # Helpful debug output for common architecture errors (size mismatch / conv)
            if "size mismatch" in str(e) or "convolution" in str(e):
                print(f"\nModel architecture error at batch {batch_idx}::")
                print(f"Input tensor x shape: {x.shape}")
               # This often happens when flattening/pooling dimensions don't match expectations
            raise e

    # Compute averaged training loss and accuracy (protect against division by zero)
    train_loss = total_loss / total if total > 0 else 0
    train_acc = correct / total if total > 0 else 0

    # Validation
    model.eval()  # set model to evaluation mode
    vloss = 0.0
    vcorrect = 0
    vtotal = 0

    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)

            batch_size = x.size(0)
            vtotal += batch_size
            vloss += loss.item() * batch_size
            vcorrect += (out.argmax(1) == y).sum().item()

    # Compute averaged validation metrics
    val_loss = vloss / vtotal if vtotal > 0 else 0
    val_acc = vcorrect / vtotal if vtotal > 0 else 0

    epoch_time = time.time() - t0

    # Record metrics for this epoch
    metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "time_s": epoch_time
    }
    history.append(metrics)

    # Print a concise summary for the epoch
    print(f"Epoch {epoch}/{EPOCHS}: "
          f"loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | "
          f"{epoch_time:.1f}s")

# Ensure the directories for MODEL_PATH and HISTORY_CSV exist
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
# Persist model weights
torch.save(model.state_dict(), MODEL_PATH)
# Save training history as CSV for later analysis
df_history = pd.DataFrame(history)
df_history.to_csv(HISTORY_CSV, index=False)

print(f"\nTraining finished!")
print(f"Model saved to: {MODEL_PATH}")
print(f"History saved to: {HISTORY_CSV}")

### Save Model Metadata and Export Artifacts to Google Drive

The script saves a JSON manifest containing model and training metadata, then copies the local artifacts folder (data_storage) to Google Drive—deleting any existing version first—and prints the location of the copied files.

In [ ]:
# Build a small manifest describing the trained model and run
manifest = {
    "model_file": MODEL_PATH,  # path to saved model weights
    "classes": genres,         # list of class names
    "seed": SEED,              # RNG seed used
    "train_epochs": EPOCHS,    # number of training epochs
    "training_time_s": sum(h["time_s"] for h in history) if history else 0 # total training time
}
# Save manifest to a JSON file inside the local data folder
savep = os.path.join(LOCAL_DATA, "model_architecture_meta.json")
import json
with open(savep, "w") as f: json.dump(manifest, f, indent=2)
# copy local data_storage to Drive (overwrite)
dst = os.path.join(DRIVE_ROOT, "waveform_analysis_outputs/data_storage")
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree(LOCAL_DATA, dst)
print("Copied artifacts to Drive:", dst)

### Summary: Training & Modeling Notebook

This notebook handles the end-to-end model development pipeline for music genre classification. Key steps include:

- **Environment & Data Sync:** Mounts Google Drive and uses automated scripts to restore datasets and audio archives to the local Colab workspace.
- **Audio Preprocessing:** Extracts specific MP3 files from the FMA archive and converts them into normalized Mel-spectrograms ready for neural network input.
- **Architecture & Data Loading:** Defines a `SmallCNN` model for feature extraction and a custom `AudioDataset` class for efficient batch processing.
- **Model Training:** Executes a robust training and validation loop with real-time metric tracking, error diagnostics, and history logging.
- **Artifact Export:** Saves the trained model weights, training history, and a metadata manifest, then synchronizes all results back to Google Drive for persistence.